In [2]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import seaborn as sns

Matplotlib is building the font cache; this may take a moment.


In [4]:
df = pd.read_csv('/Users/aalokya22gmail.com/Downloads/Timely_and_Effective_Care-Hospital.csv')

print("File loaded!")
print("Shape:", df.shape)

File loaded!
Shape: (138129, 16)


/var/folders/0c/qvpsw2jj5lz2fwjhgy56m91w0000gn/T/ipykernel_3600/2738363688.py:1: DtypeWarning: Columns (0: Facility ID) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/aalokya22gmail.com/Downloads/Timely_and_Effective_Care-Hospital.csv')


In [5]:
df.head()

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Condition,Measure ID,Measure Name,Score,Sample,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Emergency Department,EDV,Emergency department volume,very high,NaN,NaN,01/01/2024,12/31/2024
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS,Global Malnutrition Composite Score,Not Available,Not Available,5,01/01/2024,12/31/2024
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Malnutrition_Diagnosis_Documented,Global Malnutrition Composite Score: Malnutrit...,Not Available,Not Available,5,01/01/2024,12/31/2024
3,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Malnutrition_Screening,Global Malnutrition Composite Score: Malnutrit...,Not Available,Not Available,5,01/01/2024,12/31/2024
4,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Electronic Clinical Quality Measure,GMCS_Nutrition_Assessment,Global Malnutrition Composite Score: Nutrition...,Not Available,Not Available,5,01/01/2024,12/31/2024


In [6]:
df.columns.tolist()

['Facility ID',
 'Facility Name',
 'Address',
 'City/Town',
 'State',
 'ZIP Code',
 'County/Parish',
 'Telephone Number',
 'Condition',
 'Measure ID',
 'Measure Name',
 'Score',
 'Sample',
 'Footnote',
 'Start Date',
 'End Date']

In [7]:
df['Condition'].value_counts()

Condition
Electronic Clinical Quality Measure    68274
Emergency Department                   32599
Sepsis Care                            23285
Healthcare Personnel Vaccination        4657
Colonoscopy care                        4657
Cataract surgery outcome                4657
Name: count, dtype: int64

In [8]:
# Filter to only Emergency Department and Sepsis Care
df_filtered = df[df['Condition'].isin(['Emergency Department', 'Sepsis Care'])]

print("Filtered shape:", df_filtered.shape)
print("\nConditions in our dataset:")
print(df_filtered['Condition'].value_counts())

Filtered shape: (55884, 16)

Conditions in our dataset:
Condition
Emergency Department    32599
Sepsis Care             23285
Name: count, dtype: int64


In [9]:
# Let's understand the Score column
print("Score column data type:", df_filtered['Score'].dtype)
print("\nSample of Score values:")
print(df_filtered['Score'].value_counts().head(20))
print("\nHow many empty scores?", df_filtered['Score'].isna().sum())

Score column data type: str

Sample of Score values:
Score
Not Available    19402
low               1672
1                 1341
medium             917
0                  904
2                  772
very high          704
100                704
high               553
93                 468
92                 449
94                 449
86                 447
95                 445
96                 440
88                 438
91                 425
89                 423
97                 400
83                 400
Name: count, dtype: int64

How many empty scores? 0


In [10]:
# Step 2: Data Cleaning
# Replace 'Not Available' and text values with NaN (empty/null)

df_filtered = df_filtered.copy()

df_filtered['Score_clean'] = pd.to_numeric(df_filtered['Score'], errors='coerce')

print("Original Score column (text):", df_filtered['Score'].dtype)
print("New Score_clean column (numbers):", df_filtered['Score_clean'].dtype)
print("\nHow many rows became empty after cleaning?", df_filtered['Score_clean'].isna().sum())
print("How many rows have valid numbers?", df_filtered['Score_clean'].notna().sum())

Original Score column (text): str
New Score_clean column (numbers): float64

How many rows became empty after cleaning? 23248
How many rows have valid numbers? 32636


In [11]:
# Drop rows where Score_clean is empty - we can't use them for analysis
df_clean = df_filtered.dropna(subset=['Score_clean'])

# Also clean up column names - remove spaces to make them easier to type
df_clean.columns = df_clean.columns.str.strip().str.replace(' ', '_').str.replace('/', '_')

# Check our final clean dataset
print("Final clean dataset shape:", df_clean.shape)
print("\nColumn names:")
print(df_clean.columns.tolist())
print("\nSample of clean data:")
print(df_clean[['Facility_Name', 'State', 'Condition', 'Measure_Name', 'Score_clean']].head())

Final clean dataset shape: (32636, 17)

Column names:
['Facility_ID', 'Facility_Name', 'Address', 'City_Town', 'State', 'ZIP_Code', 'County_Parish', 'Telephone_Number', 'Condition', 'Measure_ID', 'Measure_Name', 'Score', 'Sample', 'Footnote', 'Start_Date', 'End_Date', 'Score_clean']

Sample of clean data:
                      Facility_Name State             Condition  \
10  SOUTHEAST HEALTH MEDICAL CENTER    AL  Emergency Department   
11  SOUTHEAST HEALTH MEDICAL CENTER    AL  Emergency Department   
14  SOUTHEAST HEALTH MEDICAL CENTER    AL  Emergency Department   
15  SOUTHEAST HEALTH MEDICAL CENTER    AL  Emergency Department   
20  SOUTHEAST HEALTH MEDICAL CENTER    AL           Sepsis Care   

                                         Measure_Name  Score_clean  
10  Average (median) time all patients spent in th...        206.0  
11  Average (median) time patients spent in the em...        208.0  
14                             Left before being seen          3.0  
15            

In [12]:
# Save our clean dataset into our project folder
df_clean.to_csv('/Users/aalokya22gmail.com/Downloads/healthcare_oper_tracker/healthcare_clean.csv', index=False)

print("✅ Clean data saved successfully!")
print("File location: /Users/aalokya22gmail.com/Downloads/healthcare_oper_tracker/healthcare_clean.csv")
print(f"Total clean records saved: {len(df_clean):,}")

✅ Clean data saved successfully!
File location: /Users/aalokya22gmail.com/Downloads/healthcare_oper_tracker/healthcare_clean.csv
Total clean records saved: 32,636


In [13]:
# Step 3: Analysis
# Question 1: Which states have the worst average ER wait times?

# First, filter to ONLY Emergency Department rows
df_ed = df_clean[df_clean['Condition'] == 'Emergency Department']

# Then filter to ONLY the wait time measure
# OP_18b measures median time patients spend in the ER
df_wait = df_ed[df_ed['Measure_ID'] == 'OP_18b']

print("Number of hospitals reporting ER wait times:", len(df_wait))
print("\nSample of wait time data:")
print(df_wait[['Facility_Name', 'State', 'Score_clean']].head(10))
print("\nNational average ER wait time (minutes):", round(df_wait['Score_clean'].mean(), 1))
print("Shortest wait time (minutes):", df_wait['Score_clean'].min())
print("Longest wait time (minutes):", df_wait['Score_clean'].max())

Number of hospitals reporting ER wait times: 4070

Sample of wait time data:
                       Facility_Name State  Score_clean
11   SOUTHEAST HEALTH MEDICAL CENTER    AL        208.0
41          MARSHALL MEDICAL CENTERS    AL        137.0
71      NORTH ALABAMA MEDICAL CENTER    AL        146.0
101         MIZELL MEMORIAL HOSPITAL    AL        121.0
131      CRENSHAW COMMUNITY HOSPITAL    AL        109.0
161               ST. VINCENT'S EAST    AL        158.0
191   DEKALB REGIONAL MEDICAL CENTER    AL        156.0
221    SHELBY BAPTIST MEDICAL CENTER    AL        190.0
251            CALLAHAN EYE HOSPITAL    AL        124.0
281            HELEN KELLER HOSPITAL    AL        184.0

National average ER wait time (minutes): 157.5
Shortest wait time (minutes): 38.0
Longest wait time (minutes): 456.0


In [14]:
# Which states have the worst average ER wait times?
state_wait = df_wait.groupby('State')['Score_clean'].mean().round(1).sort_values(ascending=False)

print("Top 10 WORST states for ER wait times (minutes):")
print(state_wait.head(10))

print("\nTop 10 BEST states for ER wait times (minutes):")
print(state_wait.tail(10))

Top 10 WORST states for ER wait times (minutes):
State
DC    326.5
PR    306.3
MD    247.7
MA    225.1
RI    223.4
DE    219.2
NY    200.3
CT    196.9
PA    186.9
NJ    185.5
Name: Score_clean, dtype: float64

Top 10 BEST states for ER wait times (minutes):
State
ID    130.0
WY    129.3
IA    126.7
KS    122.3
OK    121.7
MT    120.4
HI    118.9
NE    118.3
ND    114.9
SD    114.4
Name: Score_clean, dtype: float64


In [15]:
# Question 2: How well are hospitals treating Sepsis?
# SEP_1 measures the % of sepsis patients receiving appropriate care
df_sepsis = df_clean[df_clean['Condition'] == 'Sepsis Care']
df_sep1 = df_sepsis[df_sepsis['Measure_ID'] == 'SEP_1']

print("Hospitals reporting Sepsis care:", len(df_sep1))
print("\nNational average sepsis care compliance (%):", round(df_sep1['Score_clean'].mean(), 1))
print("Best hospital (%):", df_sep1['Score_clean'].max())
print("Worst hospital (%):", df_sep1['Score_clean'].min())

# States with worst sepsis care
state_sepsis = df_sep1.groupby('State')['Score_clean'].mean().round(1).sort_values()
print("\n10 States with LOWEST sepsis care compliance (%):")
print(state_sepsis.head(10))

Hospitals reporting Sepsis care: 3105

National average sepsis care compliance (%): 62.5
Best hospital (%): 100.0
Worst hospital (%): 0.0

10 States with LOWEST sepsis care compliance (%):
State
PR    18.1
VI    22.0
GU    37.0
CT    52.2
NM    52.9
DC    53.7
MA    54.4
MN    54.6
DE    54.7
AZ    55.8
Name: Score_clean, dtype: float64


In [16]:
# Summary 1: ER wait times by state
state_wait_df = df_wait.groupby('State').agg(
    avg_wait_minutes=('Score_clean', 'mean'),
    hospital_count=('Facility_ID', 'count')
).round(1).reset_index()

# Summary 2: Sepsis compliance by state  
state_sepsis_df = df_sep1.groupby('State').agg(
    avg_sepsis_compliance=('Score_clean', 'mean'),
    hospital_count=('Facility_ID', 'count')
).round(1).reset_index()

# Summary 3: Hospital level detail for drill-down
hospital_detail = df_wait[['Facility_ID', 'Facility_Name', 
                            'City_Town', 'State', 
                            'Score_clean']].copy()
hospital_detail.columns = ['Facility_ID', 'Facility_Name', 
                           'City', 'State', 'ER_Wait_Minutes']

# Save all three
state_wait_df.to_csv('/Users/aalokya22gmail.com/Downloads/healthcare_oper_tracker/state_er_wait.csv', index=False)
state_sepsis_df.to_csv('/Users/aalokya22gmail.com/Downloads/healthcare_oper_tracker/state_sepsis.csv', index=False)
hospital_detail.to_csv('/Users/aalokya22gmail.com/Downloads/healthcare_oper_tracker/hospital_er_detail.csv', index=False)

print("All analysis files saved!")
print(f"State ER wait summary: {len(state_wait_df)} states")
print(f"State Sepsis summary: {len(state_sepsis_df)} states")
print(f"Hospital detail: {len(hospital_detail)} hospitals")

All analysis files saved!
State ER wait summary: 53 states
State Sepsis summary: 54 states
Hospital detail: 4070 hospitals
